# Kamień 2 – trening CNN i porównanie wariantów (E1–E8)

Uruchomienie na Kaggle:
1. New Notebook → File → Import Notebook → ten plik
2. Add Input → dataset `kmader/skin-cancer-mnist-ham10000` + Upload → `splits.csv` z kamienia 1
3. Settings: Accelerator = **GPU T4 x2**, Internet = **On** (pobranie wag ImageNet)
4. Najpierw `QUICK = True` (szybki test), potem `QUICK = False` i **Save Version → Save & Run All**

In [ ]:
import os
import random
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import balanced_accuracy_score, f1_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models
from torchvision.transforms import v2

KAGGLE = Path("/kaggle/input").exists()
DATA = Path("/kaggle/input") if KAGGLE else Path("data")
OUT = Path("/kaggle/working") if KAGGLE else Path("out")
(OUT / "models").mkdir(parents=True, exist_ok=True)

CLASSES = ["nv", "mel", "bkl", "bcc", "akiec", "vasc", "df"]
MEL = CLASSES.index("mel")
EPOCHS = 15
BATCH = 32
SEEDS = [0, 1, 2]
QUICK = False  # True = szybki test: 280 zdjęć, 1 epoka, 1 seed

if QUICK:
    EPOCHS, SEEDS = 1, [0]
device = "cuda" if torch.cuda.is_available() else "cpu"
WORKERS = 4 if device == "cuda" and os.name != "nt" else 0  # Windows + Jupyter: tylko 0
print(device, torch.__version__)

## Dane i podział

In [ ]:
meta = pd.read_csv(next(DATA.rglob("HAM10000_metadata.csv")))

split_file = next(DATA.rglob("splits.csv"), None)
if split_file:
    meta = meta.merge(pd.read_csv(split_file)[["image_id", "split"]], on="image_id")
else:  # ten sam kod co w kamieniu 1
    sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=42)
    fold = np.zeros(len(meta), dtype=int)
    for k, (_, idx) in enumerate(sgkf.split(meta, meta.dx, groups=meta.lesion_id)):
        fold[idx] = k
    meta["split"] = np.select([fold < 3, fold < 6], ["test", "val"], "train")
assert (meta.groupby("lesion_id").split.nunique() == 1).all()

# podział losowy po zdjęciach – tylko do E8
tr, rest = train_test_split(meta.index, test_size=0.3, stratify=meta.dx, random_state=42)
va, te = train_test_split(rest, test_size=0.5, stratify=meta.dx[rest], random_state=42)
meta["split_random"] = "train"
meta.loc[va, "split_random"] = "val"
meta.loc[te, "split_random"] = "test"

if QUICK:
    meta = meta.groupby("dx").sample(40, random_state=0)
meta = meta.reset_index(drop=True)
meta["label"] = meta.dx.map(CLASSES.index)

pd.crosstab(meta.dx, meta.split).loc[CLASSES, ["train", "val", "test"]]

In [ ]:
# wczytanie wszystkich zdjęć do RAM, przeskalowanych do 256x256 (ok. 2 GB, szybszy trening)
paths = {p.stem: p for p in DATA.rglob("ISIC_*.jpg")}

def load(image_id):
    return np.asarray(Image.open(paths[image_id]).convert("RGB").resize((256, 256)))

t = time.time()
with ThreadPoolExecutor(8) as ex:
    images = np.stack(list(ex.map(load, meta.image_id)))
print(images.shape, f"{time.time() - t:.0f} s")

## Augmentacje i dataset

In [ ]:
norm = [v2.ToDtype(torch.float32, scale=True), v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]
geom = [v2.RandomResizedCrop(224, scale=(0.7, 1.0)), v2.RandomHorizontalFlip(), v2.RandomVerticalFlip()]

AUGS = {
    "brak": v2.Compose([v2.Resize(224)] + norm),
    "podstawowa": v2.Compose(geom + [v2.RandomRotation(20), v2.ColorJitter(0.2, 0.2)] + norm),
    "randaugment": v2.Compose(geom + [v2.RandAugment()] + norm),
}
EVAL_TF = AUGS["brak"]


class HamDataset(Dataset):
    def __init__(self, idx, tf):
        self.idx, self.tf = np.asarray(idx), tf

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        x = torch.from_numpy(images[j]).permute(2, 0, 1)  # HWC -> CHW
        return self.tf(x), meta.label[j]

In [ ]:
# podgląd augmentacji na jednym zdjęciu
x = torch.from_numpy(images[0]).permute(2, 0, 1)
fig, axes = plt.subplots(3, 5, figsize=(10, 6))
mean, std = torch.tensor([0.485, 0.456, 0.406]), torch.tensor([0.229, 0.224, 0.225])
for row, name in zip(axes, AUGS):
    for ax in row:
        img = AUGS[name](x).permute(1, 2, 0) * std + mean
        ax.imshow(img.clamp(0, 1))
        ax.axis("off")
    row[0].set_title(name, loc="left")
plt.tight_layout()
plt.show()

## Modele

In [ ]:
class SimpleCNN(nn.Module):
    # baseline: 4 bloki conv-bn-relu-pool, uczony od zera
    def __init__(self, n_classes=7):
        super().__init__()
        layers, c_in = [], 3
        for c_out in [32, 64, 128, 256]:
            layers += [nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(), nn.MaxPool2d(2)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, n_classes))

    def forward(self, x):
        return self.head(self.features(x))


def build_model(name):
    if name == "cnn":
        return SimpleCNN()
    if name == "resnet50":
        m = models.resnet50(weights="IMAGENET1K_V2")
        m.fc = nn.Linear(m.fc.in_features, len(CLASSES))  # nowa warstwa wyjściowa: 7 klas
        return m
    if name == "efficientnet_b0":
        m = models.efficientnet_b0(weights="IMAGENET1K_V1")
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(CLASSES))
        return m
    raise ValueError(name)


for name in ["cnn", "resnet50", "efficientnet_b0"]:
    print(f"{name}: {sum(p.numel() for p in build_model(name).parameters()) / 1e6:.1f} M parametrów")

## Trening i ewaluacja

In [ ]:
def evaluate(model, loader):
    model.eval()
    probs, ys = [], []
    with torch.no_grad(), torch.autocast(device_type=device, enabled=device == "cuda"):
        for x, y in loader:
            probs.append(model(x.to(device)).float().softmax(1).cpu())
            ys.append(y)
    p, y = torch.cat(probs).numpy(), torch.cat(ys).numpy()
    pred = p.argmax(1)
    rec = recall_score(y, pred, labels=range(7), average=None, zero_division=0)
    return {
        "bal_acc": balanced_accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, labels=range(7), average="macro", zero_division=0),
        "mel_recall": rec[MEL],
        "auc": roc_auc_score(y, p, multi_class="ovr"),
        "mel_auc": roc_auc_score(y == MEL, p[:, MEL]),
        **{f"rec_{c}": r for c, r in zip(CLASSES, rec)},
    }


def run(exp, seed):
    cfg = EXPERIMENTS[exp]
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    split = "split_random" if cfg["split"] == "losowy" else "split"
    tr_idx = meta.index[meta[split] == "train"]
    va_idx = meta.index[meta[split] == "val"]
    y_tr = meta.label[tr_idx].values
    counts = np.bincount(y_tr, minlength=7)

    # niezbalansowanie: ważona strata albo częstsze losowanie rzadkich klas
    weight, sampler = None, None
    if cfg["imb"] == "wazona":
        weight = torch.tensor(len(y_tr) / (7 * counts), dtype=torch.float32, device=device)
    if cfg["imb"] == "oversampling":
        sampler = WeightedRandomSampler(1.0 / counts[y_tr], num_samples=len(y_tr), replacement=True)
    loss_fn = nn.CrossEntropyLoss(weight=weight)

    train_dl = DataLoader(HamDataset(tr_idx, AUGS[cfg["aug"]]), BATCH, shuffle=sampler is None,
                          sampler=sampler, num_workers=WORKERS, drop_last=True)
    val_dl = DataLoader(HamDataset(va_idx, EVAL_TF), 64, num_workers=WORKERS)

    model = build_model(cfg["model"]).to(device)
    lr = 1e-3 if cfg["model"] == "cnn" else 1e-4  # sieć od zera potrzebuje większego lr
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.amp.GradScaler(enabled=device == "cuda")

    best, history = None, []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device, enabled=device == "cuda"):
                loss = loss_fn(model(x), y)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        sched.step()

        m = evaluate(model, val_dl)
        history.append({"exp": exp, "seed": seed, "epoch": epoch, "loss": loss.item(), "bal_acc": m["bal_acc"]})
        if best is None or m["bal_acc"] > best["bal_acc"]:  # zapisujemy najlepszą epokę wg val
            best = {**m, "epoch": epoch}
            torch.save(model.state_dict(), OUT / "models" / f"{exp}_s{seed}.pt")
    return best, history

## Eksperymenty

In [ ]:
EXPERIMENTS = {
    "E1": dict(model="cnn",             aug="podstawowa",  imb="wazona",       split="lesion_id"),
    "E2": dict(model="resnet50",        aug="podstawowa",  imb="wazona",       split="lesion_id"),
    "E3": dict(model="efficientnet_b0", aug="podstawowa",  imb="wazona",       split="lesion_id"),
    "E4": dict(model="efficientnet_b0", aug="brak",        imb="wazona",       split="lesion_id"),
    "E5": dict(model="efficientnet_b0", aug="randaugment", imb="wazona",       split="lesion_id"),
    "E6": dict(model="efficientnet_b0", aug="podstawowa",  imb="brak",         split="lesion_id"),
    "E7": dict(model="efficientnet_b0", aug="podstawowa",  imb="oversampling", split="lesion_id"),
    "E8": dict(model="efficientnet_b0", aug="podstawowa",  imb="wazona",       split="losowy"),
}
RUN = list(EXPERIMENTS)  # można ograniczyć, np. ["E1", "E2", "E3"]
pd.DataFrame(EXPERIMENTS).T

In [ ]:
res_file, hist_file = OUT / "results.csv", OUT / "history.csv"
results = pd.read_csv(res_file) if res_file.exists() else pd.DataFrame(columns=["exp", "seed"])
history = pd.read_csv(hist_file) if hist_file.exists() else pd.DataFrame()

for exp in RUN:
    for seed in SEEDS:
        if ((results.exp == exp) & (results.seed == seed)).any():
            continue  # już policzone (np. po restarcie)
        t = time.time()
        best, hist = run(exp, seed)
        best.update(exp=exp, seed=seed, minutes=(time.time() - t) / 60, **EXPERIMENTS[exp])
        results = pd.concat([results, pd.DataFrame([best])], ignore_index=True)
        history = pd.concat([history, pd.DataFrame(hist)], ignore_index=True)
        results.to_csv(res_file, index=False)
        history.to_csv(hist_file, index=False)
        print(f"{exp} seed={seed}: bal_acc={best['bal_acc']:.3f} mel_recall={best['mel_recall']:.3f} "
              f"(epoka {best['epoch']}, {best['minutes']:.1f} min)")

## Wyniki (zbiór walidacyjny)

In [ ]:
# punkt odniesienia: "model" zawsze odpowiadający nv
y_val = meta.label[meta.split == "val"]
print(f"zawsze nv -> accuracy {(y_val == 0).mean():.3f}, balanced acc {balanced_accuracy_score(y_val, np.zeros_like(y_val)):.3f}")

METRICS = ["bal_acc", "macro_f1", "mel_recall", "auc", "mel_auc"]
fmt = lambda s: f"{s.mean():.3f} ± {s.std(ddof=0):.3f}"
table = results.groupby("exp")[METRICS].agg(fmt)
table.insert(0, "konfiguracja", [" / ".join(EXPERIMENTS[e].values()) for e in table.index])
table

In [ ]:
mean = results.groupby("exp")[METRICS].mean()
std = results.groupby("exp")[METRICS].std(ddof=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, col, title in zip(axes, ["bal_acc", "mel_recall"], ["Balanced accuracy", "Recall czerniaka"]):
    ax.bar(mean.index, mean[col], yerr=std[col], capsize=3, color="#2a78d6")
    ax.set_title(title, loc="left")
    ax.set_ylim(0, 1)
axes[0].axhline(1 / 7, color="gray", ls="--", lw=1, label="zawsze nv")
axes[0].legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# krzywe uczenia (seed 0)
fig, ax = plt.subplots(figsize=(7, 3.5))
for exp, h in history[history.seed == 0].groupby("exp"):
    ax.plot(h.epoch, h.bal_acc, label=exp, lw=2)
ax.set_xlabel("epoka")
ax.set_ylabel("val balanced accuracy")
ax.legend(frameon=False, ncol=4)
plt.show()

## Hipotezy

In [ ]:
def compare(title, exps, col="bal_acc"):
    exps = [e for e in exps if e in mean.index]
    print(f"{title}:  " + "  |  ".join(f"{e} {mean.loc[e, col]:.3f}" for e in exps))

compare("H1 transfer learning vs od zera (bal_acc)", ["E1", "E2", "E3"])
compare("H2 niezbalansowanie (recall mel)", ["E6", "E3", "E7"], "mel_recall")
compare("H3 augmentacja (bal_acc)", ["E4", "E3", "E5"])
compare("H4 podział losowy vs lesion_id (bal_acc)", ["E3", "E8"])

In [ ]:
# H3: recall per klasa – czy augmentacja pomaga głównie rzadkim klasom
rec_cols = [f"rec_{c}" for c in CLASSES]
results.groupby("exp")[rec_cols].mean().reindex(["E4", "E3", "E5"]).dropna().round(3)

## Wybór najlepszego modelu

In [ ]:
# tylko warianty z uczciwym podziałem (bez E8), decyduje średnie bal_acc na walidacji
best_exp = mean.drop(index="E8", errors="ignore").bal_acc.idxmax()
best_run = results[results.exp == best_exp].sort_values("bal_acc").iloc[-1]
print(f"najlepszy wariant: {best_exp} {EXPERIMENTS[best_exp]}")
print(f"checkpoint do kamienia 3: models/{best_exp}_s{int(best_run.seed)}.pt (val bal_acc {best_run.bal_acc:.3f})")

## Wnioski

(uzupełnić po uruchomieniu)

- H1:
- H2:
- H3:
- H4:
- wybrany model:

Zbiór testowy nie był używany – ocena końcowa w kamieniu 3.